# AI Co-Scientist: Blind Data and Hypotheses
# SOLUTIONS

Solutions to the challenges in `lessons/1_Blind_Data_and_Hypotheses.ipynb`. Run the setup cells first.

In [ ]:
# Imports and paths used in this notebook.
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'lessons' / 'scripts' / '02_openevolve'))
from evaluator import load_rows, log_error

In [ ]:
rows = load_rows(ROOT / 'data' / 'observations.csv')
train = [row for row in rows if row['split'] == 'train']
test = [row for row in rows if row['split'] == 'test']
print(len(rows), 'rows:', len(train), 'train and', len(test), 'test')

In [ ]:
groups = sorted({row['group'] for row in rows})
for group in groups:
    in_group = [row for row in rows if row['group'] == group]
    print(f"group {group}: {len(in_group)} rows, x2 = {in_group[0]['x2']:.4g}, split = {in_group[0]['split']}")

## 🥊 Challenge 1: The Slope Within One Group

In [ ]:
group_a = [row for row in rows if row['group'] == 'A']
log_x1 = np.log10([row['x1'] for row in group_a])
log_y = np.log10([row['y'] for row in group_a])

slope, intercept = np.polyfit(log_x1, log_y, 1)
print(f'slope p = {slope:.3f}')

## 🥊 Challenge 2: Score a Hypothesis

In [ ]:
log_c = np.mean([np.log10(row['y']) - 1.5 * np.log10(row['x1']) for row in train])


def predict(x1, x2):
    return 10 ** log_c * x1 ** 1.5


print('train error:', round(log_error(predict, train), 3), 'dex')
print('test error: ', round(log_error(predict, test), 3), 'dex')

## 🥊 Challenge 3: Run the Agent Loop by Hand

In [ ]:
# One possible answer: the agents often propose a power law in both x1 and x2.
# Fit all three constants at once with least squares in log space.
design = np.column_stack([
    np.ones(len(train)),
    np.log10([row['x1'] for row in train]),
    np.log10([row['x2'] for row in train]),
])
target = np.log10([row['y'] for row in train])
(log_c, p, q), *_ = np.linalg.lstsq(design, target, rcond=None)


def predict(x1, x2):
    return 10 ** log_c * x1 ** p * x2 ** q


print(f'y = {10 ** log_c:.3g} * x1^{p:.3f} * x2^{q:.3f}')
print('train error:', round(log_error(predict, train), 3), 'dex')
print('test error: ', round(log_error(predict, test), 3), 'dex')